In [6]:
pip install elapid

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 11.5 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [elapid]2m5/7 [rasterio]
Note: you may need to restart the kernel to use updated packages.


In [8]:
import numpy as np
import pandas as pd
import itertools
import warnings
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score
import elapid as ela
import joblib

In [10]:
CSV_PATH = Path("training_matrix_with_absences.csv") 

In [12]:
df=pd.read_csv(CSV_PATH)

In [14]:
df.head()

,x_coord,y_coord,presence,elevation,slope,aspect
0,9.370629e+05,-2.602305e+06,1,113.80469,12.086203,229.215260
1,-9.753301e+05,-2.762410e+06,1,524.99460,23.021828,28.830984
2,-9.753301e+05,-2.762410e+06,1,524.99460,23.021828,28.830984
3,-9.753301e+05,-2.762410e+06,1,524.99460,23.021828,28.830984
4,-1.131976e+06,-2.573076e+06,1,537.42487,18.381567,0.392016


In [16]:
aspect_rad = np.deg2rad(df["aspect"])

In [18]:
df["aspect_sin"]=np.sin(aspect_rad)

In [20]:
df["aspect_cos"]=np.cos(aspect_rad)

In [22]:
df.head()

,x_coord,y_coord,presence,elevation,slope,aspect,aspect_sin,aspect_cos
0,9.370629e+05,-2.602305e+06,1,113.80469,12.086203,229.215260,-0.757169,-0.653219
1,-9.753301e+05,-2.762410e+06,1,524.99460,23.021828,28.830984,0.482227,0.876046
2,-9.753301e+05,-2.762410e+06,1,524.99460,23.021828,28.830984,0.482227,0.876046
3,-9.753301e+05,-2.762410e+06,1,524.99460,23.021828,28.830984,0.482227,0.876046
4,-1.131976e+06,-2.573076e+06,1,537.42487,18.381567,0.392016,0.006842,0.999977


# 1. Separate unique presence and background locations

In [25]:
presence_sites = df[df["presence"] ==1].drop_duplicates (subset = ["x_coord","y_coord"])[["x_coord","y_coord","presence"]].copy()

In [27]:
background_sites = df[df["presence"]==0].drop_duplicates(subset=["x_coord","y_coord"])[["x_coord","y_coord","presence"]].copy()

# 2. Create five spatial folds based on presence locations

In [30]:
kmeans = KMeans(n_clusters = 5, n_init = 50, random_state = 1234)

In [32]:
presence_sites["fold"]=kmeans.fit_predict(presence_sites[["x_coord","y_coord"]])

# 3. Assign every background location to its nearest presence-based fold

In [35]:
background_sites["fold"] = kmeans.predict(background_sites[["x_coord","y_coord"]])

# 4. Combine the two location tables and check fold balance

In [38]:
unique_sites = pd.concat([presence_sites,background_sites],ignore_index = True)

In [42]:
fold_check=pd.crosstab(unique_sites["fold"],unique_sites["presence"])

In [44]:
fold_check.columns = ["Background(0)", "Presence (1)"]
print(fold_check)

      Background(0)  Presence (1)
fold                             
0                 5             6
1                20            13
2                12             6
3                12             4
4                 6             6


In [46]:
df = pd.merge(df,unique_sites [["x_coord","y_coord","fold"]],on=["x_coord","y_coord"],how="left")

In [48]:
print(pd.crosstab(df["fold"],df["presence"]))

presence   0   1
fold            
0          5   8
1         20  30
2         12   7
3         12   4
4          6   6


In [50]:
FEATURES = ["elevation","slope","aspect_sin","aspect_cos"]
X,y = df[FEATURES], df["presence"]

In [52]:
def spatial_cv_splits(df,n_folds = 5):
    idx = df.index.to_numpy()
    for test_fold in range(n_folds):
        test_mask = (df["fold"] == test_fold).to_numpy()
        yield idx[~test_mask], idx[test_mask]

In [54]:
cv_splits = list(spatial_cv_splits(df))

In [56]:
logit_pipe = Pipeline([("scaler",StandardScaler()),("logistic", LogisticRegression(max_iter=2000,random_state=1234)),])

In [58]:
logit_grid={"logistic__C":[0.01,0.1,1,10,100]}

In [60]:
logit_search = GridSearchCV(logit_pipe,logit_grid,cv=cv_splits,scoring="roc_auc",n_jobs=-1)
logit_search.fit(X,y)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...state=1234))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'logistic__C': [0.01, 0.1, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.","[(array([ 0, ...07, 108, 109]), ...), (array([ 0, ...05, 107, 108]), ...), ...]"
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose ver

In [62]:
print(f"Best params: {logit_search.best_params_}")

Best params: {'logistic__C': 0.1}


In [64]:
print(f"Best mean CV ROC-AUC: {logit_search.best_score_:.3f}")

Best mean CV ROC-AUC: 0.450


In [74]:
rf_grid = {"n_estimators": [100,200,400], "max_depth": [None,3,5,8],"min_samples_leaf": [1,2,4],"max_features": ["sqrt","log2"],}

In [78]:
rf_search =GridSearchCV(RandomForestClassifier(random_state=1234),rf_grid,cv=cv_splits, scoring="roc_auc",n_jobs = -1,)

In [80]:
rf_search.fit(X,y)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestC...om_state=1234)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'max_depth': [None, 3, ...], 'max_features': ['sqrt', 'log2'], 'min_samples_leaf': [1, 2, ...], 'n_estimators': [100, 200, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.","[(array([ 0, ...07, 108, 109]), ...), (array([ 0, ...05, 107, 108]), ...), ...]"
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity an

In [82]:
print(f"Best params: {rf_search.best_params_}")

Best params: {'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 4, 'n_estimators': 200}


In [86]:
print(f"Best mean CV ROC-AUC: {rf_search.best_score_:.3f}")

Best mean CV ROC-AUC: 0.557
